# Teste isolado — TCU (Notícias filtradas por "Solução Consensual")

Fonte candidata: **TCU — Tribunal de Contas da União**, tema "Solução
Consensual" (setor Regulatório/Múltiplo -- cobre energia, transporte,
telecom e óleo&gás na mesma fonte, dado o caráter transversal do TCU).
Notebook **descartável** (Fase 1) -- sem dispatcher, sem
`atualizar_status_fonte`, sem gravar nada. Só valida:

1. Se a listagem já vem pronta no HTML (SSR) ou precisa de truque de API
   (tipo ANATEL/ONS) -- confirma antes de assumir
2. Se existe paginação por URL além da primeira página
3. Scraping da listagem + extração de texto completo de uma notícia

## Confirmado antes de assumir

**Bloqueio de bot na requisição simples.** `curl`/`httpx` puro levam a
uma página de challenge JS (padrão Akamai Bot Manager, cookie `TSPD`).
`curl_cffi` com impersonation de TLS (`chrome120`/`123`/`124`) passa
normal, mesmos headers já usados no resto do projeto.

**SSR confirmado -- sem truque de API necessário.** Site é Next.js (App
Router, RSC streaming via `self.__next_f.push(...)`), mas diferente do
caso ANATEL (onde o HTML da listagem era só a casca do SearchBlock
React, sem itens), aqui os 15 itens da primeira página já vêm prontos no
HTML puro -- título, data (`<time datetime="...">`, já ISO 8601), resumo
e link, todos presentes num `GET` simples (com `curl_cffi`). Confere com
a checagem manual do pedido.

**Paginação confirmada -- via query string `?pagina=N`.** Testei vários
nomes de parâmetro (`page`, `p`, `pageNumber`, `pageIndex`, `offset`) --
só `pagina` (em português) muda o conteúdo. O próprio payload RSC embutido
no HTML revela os metadados exatos: `pageSize: 15, totalElements: 93,
totalPages: 7` -- bate com os 93 itens da checagem manual do pedido.
Percorrendo `pagina=1` até `pagina=7` (preservando o `tema=` no query
string) confirma exatamente 93 URLs únicas (15×6 + 3 na última página),
sem sobreposição entre páginas.

Diferente de ABAR/AESBE, aqui **não há duplicação** -- cada página tem
exatamente 15 âncoras (ou 3 na última), todas com hrefs únicos, sem
widget "hero" nem repetição de variantes responsivas.

**Texto completo**: `soup.find("article")` (primeiro `<article>` da
página) já dá texto limpo (título, subtítulo, data, resumo, corpo) --
cai no fallback já existente em `extrair_texto_generico()` do dispatcher
compartilhado (quando nenhum seletor de `SELETORES_CONTEUDO` bate, tenta
`article` antes de usar a página inteira) -- não precisa de seletor novo
nem de extrator próprio.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
import urllib.parse
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

TEMA = "Solução consensual"
SITE_URL = f"https://portal.tcu.gov.br/imprensa/noticias?tema={urllib.parse.quote(TEMA)}"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 2000:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 2000:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação)

`a[href^="/imprensa/noticias/"]` contendo um `article` -- título em
`h2`, data em `time[datetime]` (já ISO 8601, sem regex necessário),
resumo em `p` (não usado nos metadados, só pra conferência). Paginação
via `&pagina=N`, preservando o `tema=`.

In [0]:
def listar_tcu_consenso(max_paginas: int = 7) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = f"{SITE_URL}&pagina={pagina}"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página {pagina}; parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        anchors = soup.select('a[href^="/imprensa/noticias/"]')
        anchors = [a for a in anchors if a.select_one("article")]
        if not anchors:
            print(f"  -> nenhum item encontrado na página {pagina}; fim da listagem.")
            break

        novos_na_pagina = 0
        for a in anchors:
            url_item = urllib.parse.urljoin("https://portal.tcu.gov.br", a["href"].strip())
            if url_item in vistos:
                continue
            vistos.add(url_item)
            novos_na_pagina += 1

            h2 = a.select_one("h2")
            titulo = h2.get_text(strip=True) if h2 else None
            if not titulo:
                continue

            data_publicacao = None
            tag_time = a.select_one("time[datetime]")
            if tag_time and tag_time.get("datetime"):
                data_publicacao = tag_time["datetime"][:10]

            resumo = None
            tag_p = a.select_one("p")
            if tag_p:
                resumo = tag_p.get_text(" ", strip=True)

            itens.append({
                "titulo": titulo,
                "url": url_item,
                "published_at": data_publicacao,
                "resumo": resumo,
            })

        print(f"  página {pagina}: {len(anchors)} âncoras, {novos_na_pagina} novos.")
        if not novos_na_pagina:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_tcu_consenso()

print(f"\n{len(itens)} notícias listadas.\n")
print(f"{'DATA':<12} TÍTULO")
print("-" * 100)
for item in itens:
    print(f"{item['published_at'] or '?':<12} {item['titulo'][:80]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)}")
print(f"\nExemplo de link: {itens[0]['url']}")
print(f"Exemplo de resumo: {itens[0]['resumo']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

`soup.find("article")` (primeiro `<article>` da página) -- mesmo
fallback já existente em `extrair_texto_generico()` do dispatcher
compartilhado, sem seletor novo.

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    article = soup.find("article")
    base = article if (article and len(article.get_text(strip=True)) > 500) else soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_noticia_tcu(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)

    return {
        "titulo": item["titulo"],
        "url": item["url"],
        "published_at": item["published_at"],
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_tcu(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos.")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars: {len(curtas)}")

In [0]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem já vem pronta no HTML (SSR, sem truque
de API necessário -- diferente de ANATEL/ONS), paginação confirmada via
`&pagina=N`, 93 URLs únicas em 7 páginas (bate exatamente com
`totalElements: 93` do payload RSC embutido). Texto completo sai limpo
via `article` (fallback já existente no dispatcher, sem seletor novo).
Sem duplicação de itens (diferente de ABAR/AESBE) -- cada página tem
exatamente os itens esperados, sem widget hero nem repetição interna.

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` -- sem Selenium, sem parsing que dependa de JS (o
`curl_cffi` com impersonation já é suficiente, mesmo padrão de
ABAR/CCEE). Precisa de uma `listar_tcu_consenso()` própria (paginação
`&pagina=N`, seletor `a[href^="/imprensa/noticias/"]` com `article`
filho) -- data já vem pronta na listagem (`extrair_data: None`, mesmo
padrão de ABAR/agesan_noticias), título também já vem pronto e correto
na listagem (`extrair_titulo: None`). Texto reaproveita
`extrair_texto_generico()` sem nenhuma mudança no dispatcher
compartilhado (nem seletor novo em `SELETORES_CONTEUDO`, nem função de
extração própria) -- o fallback pra `article` já cobre esse caso.

Fonte pequena (93 itens / 7 páginas) -- diferente de ANP/ABEGÁS/ABAR
(históricos grandes, capturados parcialmente), aqui dá pra cobrir o
arquivo inteiro numa execução (`max_paginas=7`).

**Sobre o registro**: já existe uma entrada "SECEX CONSENSO" no
catálogo (source_id "—", nunca implementada) -- UPDATE nela na Fase 3,
sem criar linha nova com outro nome.